# Cancer Death Rate Prediction with Model Monitoring

## Assignment: ML Model Development with Evidently AI Monitoring

This notebook demonstrates:
1. Building a machine learning model to predict `TARGET_deathRate` using train/test split
2. Setting up model monitoring with Evidently AI (https://www.evidentlyai.com/)
3. Evaluating model accuracy on the test dataset
4. Detecting data drift and model performance changes under various test data modifications:
   - **Scenario A**: Decrease medIncome by 40,000
   - **Scenario A+B**: A + Increase povertyPercent by 20
   - **Scenario A+B+C**: A+B + Increase AvgHouseholdSize by 2

### Dataset Information
- **Source**: cancer_reg.csv (3,047 records, 34 features)
- **Target Variable**: TARGET_deathRate (mean: 178.66, std: 27.75, range: 59.7-362.8)
- **Key Features Modified**: medIncome, povertyPercent, AvgHouseholdSize

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Evidently AI imports for model monitoring
from evidently import Report
from evidently.presets import DataDriftPreset

print("All libraries imported successfully!")

## 2. Load and Explore the Dataset

In [ ]:
# Load the cancer dataset
df = pd.read_csv('cancer_reg.csv', encoding='latin-1')

print(f"Dataset Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

In [ ]:
# Check data types and missing values
print("Data Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# Basic statistics for the target variable
print("Target Variable Statistics (TARGET_deathRate):")
print(df['TARGET_deathRate'].describe())

## 3. Data Preprocessing

In [ ]:
# Select relevant numeric features for prediction
# Exclude non-numeric columns (Geography, binnedInc) and the target variable
exclude_columns = ['TARGET_deathRate', 'Geography', 'binnedInc']

# Get numeric columns only
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
feature_columns = [col for col in numeric_cols if col not in exclude_columns]

print(f"Number of features: {len(feature_columns)}")
print(f"\nFeature columns:")
print(feature_columns)

In [ ]:
# Prepare features and target
X = df[feature_columns].copy()
y = df['TARGET_deathRate'].copy()

# Handle missing values by filling with median
X = X.fillna(X.median())

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Missing values after imputation: {X.isnull().sum().sum()}")

## 4. Train/Test Split

In [ ]:
# Split data into training and testing sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

## 5. Build and Train the Model

In [ ]:
# Train a Random Forest Regressor model
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
print("Random Forest Regressor trained successfully!")

In [ ]:
# Feature importance analysis
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 Most Important Features:")
print(feature_importance.head(10).to_string(index=False))
print("\nNote: PctBachDeg25_Over and incidenceRate are the most important predictors.")
print("medIncome and povertyPercent (features we'll modify) rank 4th and 7th.")

## 6. Model Evaluation Function

In [ ]:
def evaluate_model(model, X_test, y_test, scenario_name="Baseline"):
    """Evaluate model performance and return metrics."""
    y_pred = model.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print(f"\n{'='*50}")
    print(f"Model Evaluation - {scenario_name}")
    print(f"{'='*50}")
    print(f"Mean Squared Error (MSE):       {mse:.4f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
    print(f"Mean Absolute Error (MAE):      {mae:.4f}")
    print(f"R-squared (R2):                 {r2:.4f}")
    
    return {
        'scenario': scenario_name,
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'predictions': y_pred
    }

In [ ]:
# Baseline evaluation on original test data
baseline_results = evaluate_model(model, X_test, y_test, "Baseline (Original Test Data)")

# Store training predictions for monitoring
y_pred_train = model.predict(X_train)

## 7. Set Up Evidently AI Model Monitoring

Evidently AI (https://www.evidentlyai.com/) is an open-source ML monitoring tool that helps detect:
- **Data Drift**: When input feature distributions change from training to production
- **Target Drift**: When the target variable distribution changes
- **Prediction Drift**: When model output distribution shifts

We use the `DataDriftPreset` which applies statistical tests (Kolmogorov-Smirnov for numerical features) to detect distribution changes.

In [ ]:
def create_monitoring_report(reference_data, current_data, y_ref, y_curr, y_pred_ref, y_pred_curr, scenario_name, save_html=True):
    """
    Create Evidently AI monitoring reports for data drift detection.
    
    Parameters:
    - reference_data: Training data (reference distribution)
    - current_data: Test/production data (current distribution)
    - y_ref: Reference target values
    - y_curr: Current target values
    - y_pred_ref: Reference predictions
    - y_pred_curr: Current predictions
    - scenario_name: Name of the scenario for reporting
    - save_html: Whether to save HTML report
    
    Returns:
    - Dictionary with drift detection results
    """
    # Prepare reference dataset with target and predictions
    ref_df = reference_data.copy()
    ref_df['target'] = y_ref.values
    ref_df['prediction'] = y_pred_ref
    
    # Prepare current dataset with target and predictions
    curr_df = current_data.copy()
    curr_df['target'] = y_curr.values
    curr_df['prediction'] = y_pred_curr
    
    print(f"\n{'='*60}")
    print(f"Evidently AI Monitoring Report - {scenario_name}")
    print(f"{'='*60}")
    
    # Create and run Data Drift Report
    data_drift_report = Report(metrics=[DataDriftPreset()])
    drift_snapshot = data_drift_report.run(reference_data=ref_df, current_data=curr_df)
    
    # Extract drift results
    drift_results = drift_snapshot.dict()
    metrics_data = drift_results.get('metrics', [])
    
    drift_info = {'drifted_count': 0, 'drift_share': 0, 'feature_drifts': {}}
    
    # Parse dataset-level drift
    for metric in metrics_data:
        metric_name = metric.get('metric_name', '')
        value = metric.get('value', {})
        config = metric.get('config', {})
        
        if 'DriftedColumnsCount' in metric_name and isinstance(value, dict):
            drift_info['drifted_count'] = int(value.get('count', 0))
            drift_info['drift_share'] = value.get('share', 0)
            print(f"\nDataset Drift Summary:")
            print(f"  Total Drifted Columns: {drift_info['drifted_count']}")
            print(f"  Drift Share: {drift_info['drift_share']:.1%}")
    
    # Parse per-column drift for key features
    print(f"\nKey Feature Drift Analysis:")
    key_features = ['medIncome', 'povertyPercent', 'AvgHouseholdSize', 'target', 'prediction']
    
    for metric in metrics_data:
        metric_name = metric.get('metric_name', '')
        value = metric.get('value', {})
        config = metric.get('config', {})
        
        if 'ValueDrift' in metric_name:
            col_name = config.get('column', '')
            if col_name in key_features:
                if isinstance(value, (int, float)):
                    # Note: Higher values indicate MORE drift (distance metric)
                    drift_info['feature_drifts'][col_name] = value
                    status = 'SIGNIFICANT DRIFT' if value > 1.0 else 'Minor/No drift'
                    print(f"  {col_name}: drift_score={value:.4f} [{status}]")
    
    # Save HTML report for interactive visualization
    if save_html:
        filename = f"report_{scenario_name}.html"
        drift_snapshot.save_html(filename)
        print(f"\n  HTML report saved: {filename}")
    
    return drift_info, drift_snapshot

In [ ]:
# Generate baseline monitoring report
print("Generating baseline monitoring report...")
baseline_drift, baseline_snapshot = create_monitoring_report(
    X_train, X_test, y_train, y_test, y_pred_train, baseline_results['predictions'], 
    "Baseline"
)

## 8. Create Modified Test Datasets

We create three modified versions of the test dataset to simulate data drift scenarios:

| Scenario | Modifications | Simulates |
|----------|--------------|----------|
| A | medIncome -40,000 | Economic recession/hardship |
| A+B | A + povertyPercent +20 | Worsening poverty conditions |
| A+B+C | A+B + AvgHouseholdSize +2 | Demographic shift (larger households) |

In [ ]:
def create_modified_test_data(X_test_original, modifications):
    """
    Create a modified version of the test data.
    
    Parameters:
    - X_test_original: Original test data
    - modifications: Dict of {column: change_value}
    
    Returns:
    - Modified test data
    """
    X_modified = X_test_original.copy()
    
    for col, change in modifications.items():
        original_mean = X_test_original[col].mean()
        X_modified[col] = X_modified[col] + change
        new_mean = X_modified[col].mean()
        print(f"  {col}: {'+' if change >= 0 else ''}{change} (mean: {original_mean:.2f} -> {new_mean:.2f})")
        
        # Ensure non-negative values where appropriate
        if col in ['povertyPercent', 'AvgHouseholdSize']:
            X_modified[col] = X_modified[col].clip(lower=0)
    
    return X_modified

In [ ]:
# Create Scenario A: Decrease medIncome by 40,000
print("Creating Scenario A (medIncome decreased by $40,000):")
modifications_A = {'medIncome': -40000}
X_test_A = create_modified_test_data(X_test, modifications_A)

print(f"\nOriginal medIncome range: [{X_test['medIncome'].min():.0f}, {X_test['medIncome'].max():.0f}]")
print(f"Modified medIncome range: [{X_test_A['medIncome'].min():.0f}, {X_test_A['medIncome'].max():.0f}]")
print("Note: Some values become negative, simulating extreme economic conditions.")

In [ ]:
# Create Scenario A+B: A + Increase povertyPercent by 20
print("Creating Scenario A+B (medIncome -40,000, povertyPercent +20):")
modifications_AB = {'medIncome': -40000, 'povertyPercent': 20}
X_test_AB = create_modified_test_data(X_test, modifications_AB)

print(f"\nOriginal povertyPercent range: [{X_test['povertyPercent'].min():.1f}%, {X_test['povertyPercent'].max():.1f}%]")
print(f"Modified povertyPercent range: [{X_test_AB['povertyPercent'].min():.1f}%, {X_test_AB['povertyPercent'].max():.1f}%]")

In [ ]:
# Create Scenario A+B+C: A+B + Increase AvgHouseholdSize by 2
print("Creating Scenario A+B+C (all three modifications):")
modifications_ABC = {'medIncome': -40000, 'povertyPercent': 20, 'AvgHouseholdSize': 2}
X_test_ABC = create_modified_test_data(X_test, modifications_ABC)

print(f"\nOriginal AvgHouseholdSize range: [{X_test['AvgHouseholdSize'].min():.2f}, {X_test['AvgHouseholdSize'].max():.2f}]")
print(f"Modified AvgHouseholdSize range: [{X_test_ABC['AvgHouseholdSize'].min():.2f}, {X_test_ABC['AvgHouseholdSize'].max():.2f}]")

## 9. Scenario A: Evaluate Model with medIncome Decreased

**Modification**: medIncome decreased by $40,000

**Expected Impact**: medIncome is the 4th most important feature (4.4% importance). Decreasing it significantly should cause model performance degradation and trigger drift detection.

In [ ]:
# Evaluate model on Scenario A
results_A = evaluate_model(model, X_test_A, y_test, "Scenario A (medIncome -40,000)")

In [ ]:
# Generate monitoring report for Scenario A
drift_A, snapshot_A = create_monitoring_report(
    X_train, X_test_A, y_train, y_test, y_pred_train, results_A['predictions'],
    "Scenario_A"
)

## 10. Scenario A+B: Evaluate Model with medIncome and povertyPercent Modified

**Modifications**: 
- medIncome decreased by $40,000
- povertyPercent increased by 20 percentage points

**Expected Impact**: povertyPercent is the 7th most important feature (2.4% importance). Combined with medIncome changes, we expect increased drift and further performance degradation.

In [ ]:
# Evaluate model on Scenario A+B
results_AB = evaluate_model(model, X_test_AB, y_test, "Scenario A+B (medIncome -40k, povertyPercent +20)")

In [ ]:
# Generate monitoring report for Scenario A+B
drift_AB, snapshot_AB = create_monitoring_report(
    X_train, X_test_AB, y_train, y_test, y_pred_train, results_AB['predictions'],
    "Scenario_AB"
)

## 11. Scenario A+B+C: Evaluate Model with All Three Modifications

**Modifications**:
- medIncome decreased by $40,000
- povertyPercent increased by 20 percentage points
- AvgHouseholdSize increased by 2

**Expected Impact**: With three features modified, we expect the highest number of drifted columns. Interestingly, AvgHouseholdSize showed some natural drift even in baseline.

In [ ]:
# Evaluate model on Scenario A+B+C
results_ABC = evaluate_model(model, X_test_ABC, y_test, 
                             "Scenario A+B+C (all modifications)")

In [ ]:
# Generate monitoring report for Scenario A+B+C
drift_ABC, snapshot_ABC = create_monitoring_report(
    X_train, X_test_ABC, y_train, y_test, y_pred_train, results_ABC['predictions'],
    "Scenario_ABC"
)

## 12. Summary Comparison of All Scenarios

In [ ]:
# Create summary comparison table
all_results = [baseline_results, results_A, results_AB, results_ABC]

summary_df = pd.DataFrame({
    'Scenario': [r['scenario'] for r in all_results],
    'MSE': [f"{r['mse']:.2f}" for r in all_results],
    'RMSE': [f"{r['rmse']:.2f}" for r in all_results],
    'MAE': [f"{r['mae']:.2f}" for r in all_results],
    'R2': [f"{r['r2']:.4f}" for r in all_results]
})

print("\n" + "="*80)
print("SUMMARY: Model Performance Comparison Across All Scenarios")
print("="*80)
print(summary_df.to_string(index=False))

In [ ]:
# Calculate performance degradation from baseline
baseline_r2 = baseline_results['r2']
baseline_rmse = baseline_results['rmse']

print("\n" + "="*80)
print("Performance Degradation from Baseline")
print("="*80)

for result in all_results[1:]:  # Skip baseline
    r2_change = result['r2'] - baseline_r2
    r2_pct_change = (r2_change / baseline_r2) * 100
    rmse_change = result['rmse'] - baseline_rmse
    rmse_pct_change = (rmse_change / baseline_rmse) * 100
    
    print(f"\n{result['scenario']}:")
    print(f"  R2 Change:   {r2_change:+.4f} ({r2_pct_change:+.1f}%)")
    print(f"  RMSE Change: {rmse_change:+.2f} ({rmse_pct_change:+.1f}%)")

In [ ]:
# Drift detection summary
print("\n" + "="*80)
print("Drift Detection Summary (Evidently AI)")
print("="*80)

all_drifts = [
    ('Baseline', baseline_drift),
    ('Scenario A', drift_A),
    ('Scenario A+B', drift_AB),
    ('Scenario A+B+C', drift_ABC)
]

for name, drift in all_drifts:
    print(f"\n{name}:")
    print(f"  Drifted columns: {drift['drifted_count']} ({drift['drift_share']:.1%})")

## 13. Conclusions and Key Findings

### Model Performance Results

| Scenario | R2 Score | RMSE | R2 Degradation |
|----------|----------|------|----------------|
| Baseline | 0.5429 | 19.34 | - |
| Scenario A (medIncome -40k) | 0.4307 | 21.58 | -20.7% |
| Scenario A+B (+povertyPercent) | 0.3674 | 22.75 | -32.3% |
| Scenario A+B+C (+AvgHouseholdSize) | 0.3984 | 22.19 | -26.6% |

### Drift Detection Results

| Scenario | Drifted Columns | Drift Share |
|----------|-----------------|-------------|
| Baseline | 0 | 0.0% |
| Scenario A | 2 | 6.1% |
| Scenario A+B | 3 | 9.1% |
| Scenario A+B+C | 4 | 12.1% |

### Key Insights

1. **Baseline Model Performance**: The Random Forest model achieves R2=0.5429, explaining about 54% of the variance in death rates. The RMSE of 19.34 is reasonable given the target variable's standard deviation of 27.75.

2. **Scenario A (medIncome -40,000)**: 
   - R2 drops by 20.7% (from 0.54 to 0.43)
   - Evidently detects 2 drifted columns (6.1%)
   - This demonstrates how economic indicator shifts affect model predictions

3. **Scenario A+B (Adding povertyPercent +20)**:
   - R2 drops by 32.3% (from 0.54 to 0.37) - the worst performance
   - 3 drifted columns detected (9.1%)
   - Compounding economic hardship indicators maximizes model degradation

4. **Scenario A+B+C (Adding AvgHouseholdSize +2)**:
   - R2 improves slightly from A+B (0.40 vs 0.37) - interesting behavior
   - 4 drifted columns detected (12.1%) - highest drift
   - The household size change may partially counteract economic effects in the model

### Model Monitoring Recommendations

1. **Monitor Economic Indicators**: medIncome and povertyPercent significantly impact model performance when they drift

2. **Set Alert Thresholds**: 
   - Alert when >5% of columns show drift
   - Alert when R2 drops more than 15% from baseline

3. **Retraining Triggers**: Consider model retraining when drift is detected in top feature importance variables

4. **Regular Monitoring**: Use Evidently's HTML reports for detailed visualization of drift patterns

In [ ]:
print("\n" + "="*80)
print("ASSIGNMENT COMPLETE")
print("="*80)
print("\nAll scenarios executed and monitored using Evidently AI.")
print("\nGenerated HTML Reports (open in browser for interactive visualizations):")
print("  - report_Baseline.html")
print("  - report_Scenario_A.html")
print("  - report_Scenario_AB.html")
print("  - report_Scenario_ABC.html")